# Накопленный охват программ в разбивке по платформам и Playback
Пример расчета накопленных охватов трансляций футбольных матчей в разрезе платформ и Playback

## Описание задачи и условий расчета
- Big TV Россия 0+
- Период: июль 2024
- ЦА: Все 4+
- Место просмотра: все места (Дом, Дача, Вне Дома)
- Каналы: Национальная телекомпания МАТЧ ТВ
- Платформа: ТВ; Десктоп; Мобайл; Десктоп, Мобайл; Все платформы
- Playback: Consolidated; Live + VOSDAL; Playback 2-7
- Программы: Футбол. Чемпионат Европы 2024
- Статистики: Quantity (тотал), CumReach(000), CumReach%

## Инициализация

При построении отчета первый шаг в любом ноутбуке - загрузка библиотек, которые помогут обращаться к TVI API и работать с данными.

Выполните следующую ячейку, для этого перейдите в нее и нажмите Ctrl+Enter

In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
import os
import re
import json
import datetime
import time
import pandas as pd
#import matplotlib.pyplot as plt
from IPython.display import JSON
from mediascope_api.core import utils

from mediascope_api.core import net as mscore
from mediascope_api.mediavortex import tasks as cwt
from mediascope_api.mediavortex import catalogs as cwc

# Настраиваем отображение

# Включаем отображение всех колонок
pd.set_option('display.max_columns', None)
# Задаем максимальное количество выводимых строк. Раскомментируйте нужную строку
# 200 строк
# pd.set_option("display.max_rows", 200)
# Отображаем все строки. ВАЖНО! Отображение большого DataFrame требует много ресурсов
# pd.set_option("display.max_rows", None)

# Cоздаем объекты для работы с TVI API
mnet = mscore.MediascopeApiNetwork()
mtask = cwt.MediaVortexTask()
cats = cwc.MediaVortexCats()

## Справочники

Получим идентификаторы, которые будут использоваться для формирования условий расчета

In [ ]:
# Справочник мест просмотра
cats.get_tv_location()

In [ ]:
# Национальная телекомпания МАТЧ ТВ
cats.get_tv_net(name=['МАТЧ ТВ'])

In [ ]:
# Программа "Футбол. Чемпионат Европы 2024"
cats.get_tv_program(name=['Футбол. Чемпионат Европы 2024'])

In [ ]:
# Справочник платформ
cats.get_tv_platform()

In [ ]:
# Справочник Playback
cats.get_tv_playbacktype()

## Формирование задания
Зададим условия расчета

In [ ]:
# Задаем период
# Период указывается в виде списка ('Начало', 'Конец') в формате 'YYYY-MM-DD'. Можно указать несколько периодов
date_filter = [('2024-07-01', '2024-07-31')]

# Задаем дни недели
weekday_filter = None

# Задаем тип дня
daytype_filter = None

# Задаем ЦА
basedemo_filter = None

# Доп фильтр ЦА, нужен только в случае расчета отношения между ЦА, например, при расчете Affinity Index
targetdemo_filter = None

# Задаем место просмотра
location_filter = None

# Задаем каналы: сеть Матч ТВ
company_filter = 'tvNetId IN (326)'

# Указываем фильтр программ: название "Футбол. Чемпионат Европы 2024"
program_filter = 'programId IN (308227)'

# Фильтр блоков
break_filter = None

# Фильтр роликов
ad_filter = None

# Задаем платформу
platform_filter = None

# Задаем playback
playbacktype_filter = None

# Указываем список срезов (группировок)
slices = [
    'researchDate', # Дата (дни)
    'programIssueDescriptionName', # Программа описание выпуска
]

# Указываем список статистик для расчета
statistics = ['QuantitySum', 'CumReach000', 'CumReachPer']

# Задаем условия сортировки
sortings = None

# Задаем опции расчета
options = {
    "kitId": 7, # Big TV
    "bigTv": True,
    "issueType": "PROGRAM", # Тип события - программы
    "baseDate": None, # Базовый день
    "baseDateCalcType": "BY_RESEARCH_PERIOD", # Автоматический базовый день - по периоду
    "oneBaseDatePerEachDateRange": False, # Автоматический базовый день - свой базовый день для каждого периода
    "useNbd": False, # НБД коррекция - выкл.
}

Сформируем словарь с местами просмотра

In [ ]:
locations = {
    'a. Дом, Дача, Вне Дома': 'locationId IN (1,2,4)',
}

Сформируем словарь с платформами

In [ ]:
platforms = {
    'a. TV, Desktop, Mobile': 'platformId IN (1,2,3)',
    'b. TV': 'platformId IN (1)',
    'c. Desktop': 'platformId IN (2)',
    'd. Mobile': 'platformId IN (3)',
    'e. Desktop, Mobile': 'platformId IN (2,3)',
}

Сформируем словарь с Playback

In [ ]:
playbacks = {
    'a. Consolidated' : 'playBackTypeId IN (0,1,2,3,4,5,6,7)',
    'b. Live + VOSDAL' : 'playBackTypeId IN (0,1)',
    'c. Playback 2-7' : 'playBackTypeId IN (2,3,4,5,6,7)',
}

Создадим комбинации

In [ ]:
combinations = utils.combine_dicts(locations, platforms, playbacks)

## Расчет задания

In [ ]:
# Посчитаем задания в цикле
tasks = []
print("Отправляем задания на расчет")

# Для каждой комбинации формируем задание и отправляем на расчет
for k, v in combinations.items():
    
    # Подставляем значения в параметры
    project_name = k
    
    location_filter = v[0] # место просмотра
    platform_filter = v[1] # платформа
    playbacktype_filter = v[2] # Playback
      
    # Формируем задание для API TV Index в формате JSON
    task_json = mtask.build_crosstab_task(date_filter=date_filter, weekday_filter=weekday_filter, daytype_filter=daytype_filter, 
                                          company_filter=company_filter, location_filter=location_filter, 
                                          basedemo_filter=basedemo_filter, targetdemo_filter=targetdemo_filter, 
                                          program_filter=program_filter, break_filter=break_filter, ad_filter=ad_filter, 
                                          platform_filter=platform_filter, playbacktype_filter=playbacktype_filter, 
                                          slices=slices, statistics=statistics, sortings=sortings, options=options)

    # Для каждого этапа цикла формируем словарь с параметрами и отправленным заданием на расчет
    tsk = {}
    tsk['project_name'] = project_name    
    tsk['task'] = mtask.send_crosstab_task(task_json)
    tasks.append(tsk)
    time.sleep(2)
    print('.', end = '')
    
print(f"\nid: {[i['task']['taskId'] for i in tasks]}") 

print('')
# Ждем выполнения
print('Ждем выполнения')
tsks = mtask.wait_task(tasks)
print('Расчет завершен, получаем результат')

# Получаем результат
results = []
print('Собираем таблицу')
for t in tasks:
    tsk = t['task'] 
    df_result = mtask.result2table(mtask.get_result(tsk), project_name = t['project_name'])        
    results.append(df_result)
    print('.', end = '')
df = pd.concat(results)

# Приводим порядок столбцов в соответствие с условиями расчета
df = df[['prj_name']+slices+statistics]

df

In [ ]:
# Разобъем столбец с комбинациями
df[['location','platform','playback']] = df['prj_name'].str.split(pat=';', n=2, expand=True)
df = df[['location','platform','playback']+slices+statistics]

df

## Настройка внешнего вида таблицы
Пропустите этот шаг, если хотите экспортировать таблицу в ее текущем виде

In [ ]:
# Формируем сводную таблицу: строки - срезы; столбцы - место просмотра, Playback, платформа; значения - статистики
df = pd.pivot_table(df, values=statistics,
                        index=slices, 
                        columns=['location','playback','platform'])
df

In [ ]:
# Опционально: переносим статистики в строки
df = df.stack(level=0).reorder_levels([2, 0, 1], axis=0).sort_index()

# Убираем объединение ячеек в строках 
df = df.reset_index()

df

## Сохраняем в Excel
По умолчанию файл сохраняется в папку `excel`

In [ ]:
df_info = mtask.task_builder.get_report_info()

with pd.ExcelWriter(mtask.task_builder.get_excel_filename('big_tv_crosstab_programs_01_sports_reach')) as writer:
    df.to_excel(writer, sheet_name='Report', index=True)
    df_info.to_excel(writer, sheet_name='Info', index=False)